# Notebook 12 — Usability and User Acceptance Analysis

**Project:** A Multi-Modal Deep Learning Approach for Eggshell Micro-Crack Morphology Analysis and Root-Cause Diagnosis in Poultry Breeder Farms
**Author:** M.P.N. Sharindi (Coventry Index 16110429)
**Stage:** 12 of 12 — answers RQ6 and completes Objective 4

---

### What this notebook does and does not do

It reads `evaluation_pack/responses.csv`, which you fill in from the facilitator notes after the
sessions, and produces Sections 5.7 and 5.8. **It contains no results and invents none.** Run it
before collecting data and it will tell you so and stop.

### What gets reported

| | |
|---|---|
| Task success | per task and overall, split into unaided / prompted / failed |
| Task time | median and range per task, with the slowest task identified |
| Discoverability | whether the Batch tab, the Model tab and the low-confidence flag were found without prompting |
| SUS | per participant, mean with 95 % CI, percentile band, per-item profile |
| Domain items | trust, agreement with the attributed cause, feasibility of actions, time saving |
| Adoption | would use / would use with changes / would not |
| Qualitative | a structured table of quotations for thematic coding |

### How to report it honestly

With ten participants the confidence interval on a SUS mean is roughly ±10 points. That is wide,
and it is not a flaw in the study — it is what a sample of ten gives you. **Report the interval,
describe the study as indicative, and do not run inferential tests between subgroups.** With this
n, a comparison between, say, experienced and inexperienced users has no power and claiming one
would be worse than saying nothing.

The task success rates are the more robust finding. "Nine of ten found the Batch tab unaided" is
a defensible sentence; "SUS was significantly above average" is not.

---
## 0. Setup

In [ ]:
import os
os.environ["EGG_ARTEFACT_ROOT"] = r"D:\Eggshell-Diagnostic-Research\Artefact"

RESPONSES = "responses.csv"      # place in Artefact/evaluation_pack/
MIN_FOR_INFERENCE = 8            # below this, report descriptively only

In [ ]:
import sys, json, warnings
from pathlib import Path
from datetime import datetime
_here = Path.cwd()
_src = next((p / "src" for p in [_here, *_here.parents] if (p / "src" / "common").is_dir()), None)
if _src is None:
    raise RuntimeError("Run from Artefact/notebooks/ or set EGG_ARTEFACT_ROOT.")
sys.path.insert(0, str(_src))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from common import project_config as CFG

warnings.filterwarnings("ignore")
STAGE = "12_usability"
OUT = CFG.results_dir(STAGE); FIG_DIR, TAB_DIR = OUT / "figures", OUT / "tables"
PACK = CFG.ARTEFACT_ROOT / "evaluation_pack"
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 10,
                     "axes.titlesize": 11, "axes.titleweight": "bold",
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "legend.frameon": False,
                     "figure.facecolor": "white", "savefig.facecolor": "white"})

path = PACK / RESPONSES
if not path.is_file():
    print(f"No responses file at {path}")
    print("\nThis notebook reports collected data. It does not simulate it.")
    print("Copy responses_template.csv to responses.csv, delete the example row, and enter one")
    print("row per participant from the facilitator notes. Then re-run.")
    raise SystemExit

df = pd.read_csv(path)
df = df[~df["participant"].astype(str).str.strip().eq("")]
df = df[~df.get("most_useful", pd.Series(dtype=str)).astype(str)
          .str.contains("example only", case=False, na=False)]
N = len(df)
if N == 0:
    print("responses.csv has no real rows yet (only the example). Nothing to report.")
    raise SystemExit
print(f"{N} participants loaded from {path.name}")
S = {"stage": STAGE, "generated_at": datetime.now().isoformat(timespec="seconds"), "n": int(N)}
if N < MIN_FOR_INFERENCE:
    print(f"\nNOTE: n = {N}. Below {MIN_FOR_INFERENCE} this study is indicative, not inferential.")
    print("Report counts and medians, quote the confidence interval, and avoid subgroup tests.")
S["indicative_only"] = bool(N < MIN_FOR_INFERENCE)

---
## 1. Participants

In [ ]:
demo_cols = [c for c in ["years_experience", "role", "software_use", "used_automated_before"]
             if c in df.columns]
if demo_cols:
    print(df[["participant"] + demo_cols].to_string(index=False))
    if "years_experience" in df:
        y = pd.to_numeric(df["years_experience"], errors="coerce").dropna()
        if len(y):
            print(f"\nexperience: median {y.median():.0f} years, range {y.min():.0f}-{y.max():.0f}")
            S["experience_median"] = float(y.median())
    for c in ["role", "software_use", "used_automated_before"]:
        if c in df:
            S[f"mix_{c}"] = df[c].value_counts().to_dict()
            print(f"{c}: {df[c].value_counts().to_dict()}")

---
## 2. Task performance

The most robust part of the study. With ten participants a count is reportable where a mean
difference is not.

In [ ]:
TASKS = {
    "t1": "Inspect a single egg", "t2": "Find crack type and cause",
    "t3": "Spot an uncertain result", "t4": "Analyse a tray",
    "t5": "Decide what to do", "t6": "Find the limitations",
}
rows = []
for k, label in TASKS.items():
    oc, tc = f"{k}_outcome", f"{k}_time_s"
    if oc not in df:
        continue
    o = df[oc].astype(str).str.upper().str.strip()
    t = pd.to_numeric(df.get(tc), errors="coerce")
    rows.append({
        "task": label,
        "unaided": int((o == "U").sum()), "prompted": int((o == "P").sum()),
        "failed": int((o == "F").sum()),
        "success_rate": round(float(((o == "U") | (o == "P")).mean()), 3),
        "unaided_rate": round(float((o == "U").mean()), 3),
        "median_s": round(float(t.median()), 1) if t.notna().any() else np.nan,
        "min_s": round(float(t.min()), 1) if t.notna().any() else np.nan,
        "max_s": round(float(t.max()), 1) if t.notna().any() else np.nan,
    })
tasks = pd.DataFrame(rows).set_index("task")
tasks.to_csv(TAB_DIR / "t01_task_performance.csv")
S["tasks"] = tasks.reset_index().to_dict("records")
print(tasks.to_string())
if len(tasks):
    worst = tasks["unaided_rate"].idxmin()
    slow = tasks["median_s"].idxmax() if tasks["median_s"].notna().any() else None
    print(f"\nlowest unaided completion: {worst} ({tasks.loc[worst, 'unaided_rate']:.0%})")
    if slow:
        print(f"slowest task: {slow} (median {tasks.loc[slow, 'median_s']:.0f} s)")
    S["worst_task"], S["slowest_task"] = str(worst), str(slow)

In [ ]:
# --- discoverability: were the key affordances found without prompting? --
DISC = {"t4_found_batch": "Found the Batch tab unaided",
        "t6_found_model": "Found the Model tab unaided",
        "t3_noticed_flag": "Noticed the low-confidence flag",
        "t2_noticed_split": "Read the root-cause percentage split",
        "t3_correct_action": "Said they would hand-check the flagged egg",
        "t5_action_specific": "Named a specific, plausable action",
        "t6_named_limitation": "Named a real limitation"}
rows = []
for c, label in DISC.items():
    if c not in df:
        continue
    v = df[c].astype(str).str.upper().str.strip().isin(["Y", "YES", "TRUE", "1"])
    rows.append({"behaviour": label, "yes": int(v.sum()), "of": int(len(v)),
                 "rate": round(float(v.mean()), 3)})
disc = pd.DataFrame(rows).set_index("behaviour")
disc.to_csv(TAB_DIR / "t02_discoverability.csv")
S["discoverability"] = disc.reset_index().to_dict("records")
print(disc.to_string())
print("\nAnything below about 0.70 is a design finding and belongs in Section 6.5 as a")
print("recommendation, not buried in the appendix.")

---
## 3. System Usability Scale

Scored the standard way: odd items contribute (response − 1), even items contribute
(5 − response), summed and multiplied by 2.5 for a 0–100 score. **It is not a percentage.**

In [ ]:
SUS_COLS = [f"sus_{i}" for i in range(1, 11)]
have = [c for c in SUS_COLS if c in df.columns]
if len(have) < 10:
    print(f"SUS incomplete: found {len(have)} of 10 items. Skipping.")
else:
    raw = df[SUS_COLS].apply(pd.to_numeric, errors="coerce")
    bad = raw[(raw < 1) | (raw > 5)].notna().any(axis=1)
    if bad.any():
        print(f"WARNING: out-of-range responses for {df.loc[bad, 'participant'].tolist()}")
    odd = raw[[f"sus_{i}" for i in (1, 3, 5, 7, 9)]] - 1
    even = 5 - raw[[f"sus_{i}" for i in (2, 4, 6, 8, 10)]]
    sus = (odd.sum(axis=1) + even.sum(axis=1)) * 2.5
    df["sus_score"] = sus

    m, sd = float(sus.mean()), float(sus.std(ddof=1)) if len(sus) > 1 else 0.0
    if len(sus) > 1:
        lo, hi = stats.t.interval(0.95, len(sus) - 1, loc=m, scale=sd / np.sqrt(len(sus)))
    else:
        lo = hi = m

    def band(x):
        if x >= 80.3: return "A — excellent (top 10%)"
        if x >= 74:   return "B — good"
        if x >= 68:   return "C — above average"
        if x >= 51:   return "D — below average"
        return "F — poor"

    S["sus"] = {"n": int(len(sus)), "mean": round(m, 1), "sd": round(sd, 1),
                "ci_low": round(float(lo), 1), "ci_high": round(float(hi), 1),
                "median": round(float(sus.median()), 1),
                "min": round(float(sus.min()), 1), "max": round(float(sus.max()), 1),
                "band": band(m), "pct_above_68": round(float((sus >= 68).mean() * 100), 1)}
    print(df[["participant", "sus_score"]].to_string(index=False))
    print(f"\nmean {m:.1f}  (95% CI {lo:.1f} to {hi:.1f}, SD {sd:.1f}, n = {len(sus)})")
    print(f"median {sus.median():.1f}, range {sus.min():.0f} to {sus.max():.0f}")
    print(f"grade band: {band(m)}")
    print(f"{(sus >= 68).mean() * 100:.0f}% of participants scored at or above 68, "
          "the usual average benchmark")
    print(f"\nThe interval is {hi - lo:.0f} points wide. Quote it. A mean alone from n = "
          f"{len(sus)} overstates what was measured.")

    item = pd.DataFrame({
        "item": [f"Q{i}" for i in range(1, 11)],
        "mean": [round(float(raw[f'sus_{i}'].mean()), 2) for i in range(1, 11)],
        "sd": [round(float(raw[f'sus_{i}'].std(ddof=1)), 2) if len(raw) > 1 else 0.0
               for i in range(1, 11)],
        "polarity": ["positive" if i % 2 else "negative" for i in range(1, 11)],
    }).set_index("item")
    item.to_csv(TAB_DIR / "t03_sus_items.csv")
    S["sus_items"] = item.reset_index().to_dict("records")
    print()
    print(item.to_string())

In [ ]:
DOMAIN = {"trust_act": "I trust the detection enough to act on it",
          "cause_matched": "The root cause matched my own conclusion",
          "actions_feasible": "The recommended actions are feasible",
          "saves_time": "This would save time versus current practice"}
rows = []
for c, label in DOMAIN.items():
    if c not in df:
        continue
    v = pd.to_numeric(df[c], errors="coerce").dropna()
    rows.append({"statement": label, "mean": round(float(v.mean()), 2),
                 "sd": round(float(v.std(ddof=1)), 2) if len(v) > 1 else 0.0,
                 "median": float(v.median()),
                 "pct_agree": round(float((v >= 4).mean() * 100), 1), "n": int(len(v))})
dom = pd.DataFrame(rows).set_index("statement")
dom.to_csv(TAB_DIR / "t04_domain_items.csv")
S["domain"] = dom.reset_index().to_dict("records")
print(dom.to_string())

if "would_use" in df:
    wu = df["would_use"].astype(str).str.strip().value_counts()
    S["would_use"] = wu.to_dict()
    print("\nwould use at their hatchery:")
    print(wu.to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
ax = axes[0]
if len(tasks):
    y = np.arange(len(tasks))
    ax.barh(y, tasks["unaided"], color="#55A868", label="unaided")
    ax.barh(y, tasks["prompted"], left=tasks["unaided"], color="#DD8452", label="prompted")
    ax.barh(y, tasks["failed"], left=tasks["unaided"] + tasks["prompted"],
            color="#C44E52", label="failed")
    ax.set_yticks(y); ax.set_yticklabels(tasks.index, fontsize=8)
    ax.invert_yaxis(); ax.set_xlabel("participants"); ax.legend(fontsize=8)
    ax.set_title("(a) Task completion")

ax = axes[1]
if "sus_score" in df:
    ax.hist(df["sus_score"], bins=np.arange(0, 105, 10), color="#4C72B0", alpha=0.85,
            edgecolor="white")
    ax.axvline(68, ls="--", color="#C44E52", lw=1.4)
    ax.text(68.8, ax.get_ylim()[1] * 0.92, "average (68)", fontsize=8, color="#C44E52")
    ax.axvline(df["sus_score"].mean(), ls="-", color="#55A868", lw=1.8)
    ax.set_xlabel("SUS score"); ax.set_ylabel("participants")
    ax.set_title(f"(b) SUS, n = {len(df)}  mean {df['sus_score'].mean():.1f}")
    ax.set_xlim(0, 100)

ax = axes[2]
if len(dom):
    ax.barh(range(len(dom)), dom["mean"], xerr=dom["sd"], color="#937860", alpha=0.9, capsize=3)
    ax.axvline(3, ls="--", color="#888", lw=1)
    ax.set_yticks(range(len(dom)))
    ax.set_yticklabels([i[:34] for i in dom.index], fontsize=8)
    ax.invert_yaxis(); ax.set_xlim(1, 5); ax.set_xlabel("mean rating (1-5)")
    ax.set_title("(c) Domain-specific items")
fig.suptitle(f"Usability evaluation with {N} poultry production managers", y=1.02, fontsize=12)
fig.tight_layout(); fig.savefig(FIG_DIR / "01_usability.png", bbox_inches="tight")
plt.show()

---
## 4. Qualitative responses

Exported as a table for thematic coding. Code it yourself, by hand, and put the coding frame in
the appendix — an unanalysed list of quotations is not a finding.

In [ ]:
qcols = [c for c in ["most_useful", "most_annoying", "change_first"] if c in df.columns]
if qcols:
    q = df[["participant"] + qcols].copy()
    q.to_csv(TAB_DIR / "t05_qualitative_responses.csv", index=False)
    for c in qcols:
        print(f"\n--- {c.replace('_', ' ').upper()} ---")
        for _, r in q.iterrows():
            txt = str(r[c]).strip()
            if txt and txt.lower() != "nan":
                print(f"  {r['participant']}: {txt}")
    print("\nSuggested coding frame to start from, then revise against the data:")
    print("  trust · speed · clarity of the explanation · actionability of advice ·")
    print("  fit with current workflow · hardware and setup · training needed")

In [ ]:
def _safe(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, (np.bool_,)): return bool(o)
    return str(o)
(OUT / "usability_summary.json").write_text(json.dumps(S, indent=2, default=_safe))

L = ["=" * 70, "NOTEBOOK 12 DIGEST -- USABILITY AND USER ACCEPTANCE", "=" * 70,
     f"generated: {S['generated_at']}", f"participants: {S['n']}"
     + ("   (INDICATIVE ONLY)" if S.get("indicative_only") else ""), ""]
if "sus" in S:
    s = S["sus"]
    L += [f"SUS mean {s['mean']} (95% CI {s['ci_low']}-{s['ci_high']}, SD {s['sd']})",
          f"    median {s['median']}, range {s['min']}-{s['max']}",
          f"    band: {s['band']}",
          f"    {s['pct_above_68']}% of participants at or above the 68 benchmark", ""]
L.append("TASK PERFORMANCE")
for r in S.get("tasks", []):
    L.append(f"   {r['task']:<30s} unaided {r['unaided']}/{N}  "
             f"success {r['success_rate']:.0%}  median {r['median_s']}s")
L += ["", "DISCOVERABILITY"]
for r in S.get("discoverability", []):
    L.append(f"   {r['behaviour']:<44s} {r['yes']}/{r['of']}  ({r['rate']:.0%})")
L += ["", "DOMAIN ITEMS (1-5)"]
for r in S.get("domain", []):
    L.append(f"   {r['statement']:<44s} {r['mean']:.2f}  {r['pct_agree']:.0f}% agree")
if "would_use" in S:
    L += ["", f"WOULD USE: {S['would_use']}"]
L.append("=" * 70)
digest = "\n".join(L)
(OUT / "digest.txt").write_text(digest)
print(digest)